# Autonomous Motion Research Hub (Google Colab Cloud GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yutotuy0209-droid/colab-motion-pipeline/blob/main/notebooks/colab_autonomous_research_hub.ipynb)

This notebook demonstrates the **5 Autonomous Research Engines** running on Google Colab:
1. **GPU-Accelerated 2D POC** (Subpixel phase correlation)
2. **Bidirectional Backlash & Hysteresis Fitting** (ISO 230-2 mechanical accuracy)
3. **Analytical G-Code Motion & Dwell Profiler** (Vibration-free timing extraction)
4. **Subpixel Grid Dot Centroid & Topology Extractor** (Top-Hat illumination flattening)
5. **Geometric QR Circle & Runout Estimator** (Prof. Li's rotation center requirement)

In [ ]:
# [Setup] Clone repository and check Cloud GPU
import os
import sys

if 'google.colab' in sys.modules:
    if not os.path.exists('colab-motion-pipeline'):
        !git clone https://github.com/yutotuy0209-droid/colab-motion-pipeline.git
        %cd colab-motion-pipeline
        sys.path.append(os.getcwd())

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Cloud GPU Device: {torch.cuda.get_device_name(0)}")

## 1. Bidirectional Backlash & Hysteresis Identification

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.hysteresis_fitter import fit_bidirectional_backlash

# Simulated forward & backward stroke with 0.8 µm mechanical backlash
cmd_fwd = np.array([0, 10, 20, 30, 40, 50], dtype=float)
cmd_bwd = np.array([50, 40, 30, 20, 10, 0], dtype=float)
cmd_all = np.concatenate([cmd_fwd, cmd_bwd])

# Forward: y = 1.002 * x + 0.1
# Backward: y = 1.002 * x - 0.7  -> Backlash gap = 0.8 µm
noise = np.random.normal(0, 0.05, len(cmd_all))
meas_all = np.where(
    np.arange(len(cmd_all)) < len(cmd_fwd),
    1.002 * cmd_all + 0.1,
    1.002 * cmd_all - 0.7
) + noise

res = fit_bidirectional_backlash(cmd_all, meas_all)
print(f"Identified Backlash Gap: {res.backlash_um:.3f} µm (Ground Truth: 0.800 µm)")
print(f"Forward Slope:  {res.alpha_forward:.4f}")
print(f"Backward Slope: {res.alpha_backward:.4f}")
print(f"3-Sigma Repeatability Uncertainty: {res.three_sigma:.3f} µm")

# Plot Hysteresis Loop
plt.figure(figsize=(7, 5))
plt.plot(cmd_all[res.forward_indices], meas_all[res.forward_indices], 'o-', color='#2563EB', label='Forward Stroke')
plt.plot(cmd_all[res.backward_indices], meas_all[res.backward_indices], 's-', color='#DC2626', label='Backward Stroke')
plt.title(f"Hysteresis Loop & Reversal Backlash [{res.backlash_um:.3f} µm]", fontsize=12, fontweight='bold')
plt.xlabel("Command Position [µm]")
plt.ylabel("Measured Displacement [µm]")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

## 2. Analytical G-Code Timeline & Camera Dwell Simulation

In [ ]:
from src.gcode_analyzer import parse_gcode_timeline

sample_nc = """
G90 G54
G01 X0.010 F1.0 ; Step 1: 10 µm
G04 X1.5       ; Dwell 1.5s for vibration relaxation
G01 X0.020 F1.0 ; Step 2: 10 µm
G04 X1.5       ; Dwell 1.5s
G01 X0.000 F2.0 ; Return
G04 X2.0
M30
"""

nc_res = parse_gcode_timeline(sample_nc)
print(f"Total Cycle Time: {nc_res.total_duration_s:.2f} s")
print(f"Total Travel Distance: {nc_res.total_distance_um}")
print(f"Detected {len(nc_res.rest_intervals)} vibration-free camera trigger windows:")
for idx, (t0, t1, pos) in enumerate(nc_res.rest_intervals, start=1):
    print(f"  Window {idx}: [{t0:.2f}s -> {t1:.2f}s] (Duration: {t1-t0:.2f}s) at X={pos['X']:.1f}µm")

## 3. Geometric QR Decomposition for Table Rotation Center & Runout

In [ ]:
from src.rotation_estimator import estimate_rotation_center_qr

# Simulated rotation arc: Center at (150, 250), Radius R=80 µm, angle 0 to 180 deg
angles = np.linspace(0, np.pi, 15)
true_cx, true_cy, true_r = 150.0, 250.0, 80.0
arc_pts = np.column_stack([
    true_cx + true_r * np.cos(angles) + np.random.normal(0, 0.2, len(angles)),
    true_cy + true_r * np.sin(angles) + np.random.normal(0, 0.2, len(angles))
])

rot_res = estimate_rotation_center_qr(arc_pts)
print(f"Estimated Rotation Center: ({rot_res.center_x:.3f}, {rot_res.center_y:.3f}) [True: ({true_cx}, {true_cy})]")
print(f"Estimated Radius:          {rot_res.radius:.3f} µm [True: {true_r}]")
print(f"Peak-to-Valley Runout:     {rot_res.peak_to_valley_runout:.3f} µm (Eccentricity error)")
print(f"Circularity RMS:           {rot_res.circularity_rms:.3f} µm")

# Circle Plot
theta_grid = np.linspace(0, 2*np.pi, 200)
plt.figure(figsize=(6, 6))
plt.plot(arc_pts[:, 0], arc_pts[:, 1], 'ro', label='Tracked Marker Positions')
plt.plot(rot_res.center_x + rot_res.radius * np.cos(theta_grid),
         rot_res.center_y + rot_res.radius * np.sin(theta_grid),
         '--', color='#2563EB', label='Fitted Circle (QR Solver)')
plt.plot(rot_res.center_x, rot_res.center_y, 'k+', markersize=12, label=f'Center ({rot_res.center_x:.1f}, {rot_res.center_y:.1f})')
plt.axis('equal')
plt.title("Geometric QR Circle Fitting & Coaxiality", fontsize=12, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()